## ML_1023: GPT-2 Transformer Block

**Difficulty**: Hard | **TorchCode**: #13

### Problem
Implement a GPT-2 style transformer block: pre-norm architecture with causal multi-head self-attention and a 4× MLP with GELU activation.

### Formula
$$x \leftarrow x + \text{CausalMHA}(\text{LN}_1(x))$$
$$x \leftarrow x + \text{MLP}(\text{LN}_2(x)), \quad \text{MLP} = W_2\,\text{GELU}(W_1 x)$$

In [ ]:
import torch
import torch.nn as nn
import math

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

    def _attn(self, x):
        B, S, _ = x.shape
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        mask = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        attn = torch.matmul(weights, v)
        return self.W_o(attn.transpose(1, 2).contiguous().view(B, S, -1))

    def forward(self, x):
        x = x + self._attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Output shape ---
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
assert isinstance(block, nn.Module)
assert block(torch.randn(2, 8, 64)).shape == (2, 8, 64)

# --- Test 2: Pre-norm architecture (has ln1 and ln2) ---
block2 = GPT2Block(d_model=32, num_heads=4)
assert isinstance(block2.ln1, nn.LayerNorm) and isinstance(block2.ln2, nn.LayerNorm)

# --- Test 3: MLP has 4× expansion ---
linears = [m for m in block2.mlp.modules() if isinstance(m, nn.Linear)]
assert linears[0].weight.shape == (128, 32)
assert linears[-1].weight.shape == (32, 128)

# --- Test 4: Causal masking ---
torch.manual_seed(0)
x = torch.randn(1, 8, 32)
out1 = block2(x)
x2 = x.clone(); x2[:, 4:] = torch.randn(1, 4, 32)
out2 = block2(x2)
assert torch.allclose(out1[:, :4], out2[:, :4], atol=1e-5)

# --- Test 5: Gradient flow to all parameters ---
torch.manual_seed(0)
x = torch.randn(1, 4, 32, requires_grad=True)
block2(x).sum().backward()
assert x.grad is not None
n_total = sum(1 for p in block2.parameters())
n_grad = sum(1 for p in block2.parameters() if p.grad is not None)
assert n_grad == n_total

print('All tests passed!')